# Severity x duration: 3 x 3 anomaly-event matrix

This notebook runs the production backend preprocessing and quantile ML pipeline, converts flagged site-months into de-duplicated events, and builds the requested matrix.

- **X axis — severity:** Low `< 1`, Medium `1 to < 3`, High `>= 3`. The continuous score is `(actual - q95) / (q95 - q05)`, so it means prediction-band widths above the learned upper bound. These cut points preserve the dashboard's existing `<1 / 1-3 / >=3` tiers while using the requested labels.
- **Y axis — duration:** Single month, 2-3 consecutive months, `>=4` consecutive months. A month remains elevated while `kWh >= 1.3 x` the median of the four complete pre-event months.
- **Unit of analysis:** one event, not one flag. Consecutive flags inside the same elevated run are merged. Duration walks backward from the first surfaced flag to an earlier qualifying start; severity is the **detection** score because an unflagged earlier month has no model score.
- **Censoring:** an open 1-3 month run at the end of history or before a missing month is not forced into a duration cell. Four observed elevated months are sufficient to confirm the `>=4` cell.

The final spike/step comparison is a consistency check of the product intuition, not proof of why usage changed. Both methods use the same billing history.

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import json
import sys

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Work whether Jupyter starts in Backend/ or Backend/notebooks/.
ROOT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
BACKEND_ROOT = next((p for p in ROOT_CANDIDATES if (p / "app" / "ml").is_dir()), None)
if BACKEND_ROOT is None:
    raise RuntimeError("Start Jupyter in Anomaly-Dashboard-Backend or its notebooks directory.")
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.data_container import DataBillContainer
from app.ml.config import (
    ClassifyThresholds, DateRange, DropOptions, QuantileConfig,
)
from app.ml.pipeline import (
    abnormal_dataframe, build_pipeline, classify_pipeline, preview_missing_rate,
)
from app.ml.severity_duration import (
    DURATION_BANDS, SEVERITY_BANDS, SeverityDurationConfig,
    build_severity_duration_events, duration_intuition_report,
    duration_type_crosstab, recompute_quantile_severity,
    severity_duration_matrix,
)
from app.ml.state import MLRunState

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print(f"Backend root: {BACKEND_ROOT}")
print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__}")

## 1. Configuration

`DATA_MODE = "auto"` uses the five local raw files when all are present and otherwise falls back to the deterministic demo. Use `"raw_files"` to require them or `"synthetic"` to force the demo. Raw mode calls `DataBillContainer.load_files`, so it uses the same PEA/MEA cleaning as the API.

In [ ]:
DATA_MODE = "auto"  # "auto", "raw_files", or "synthetic"

RAW_DATA_DIR = BACKEND_ROOT / "app" / "ml" / "data"
mea_tuc_candidates = sorted(RAW_DATA_DIR.glob("MEA_2019-2026_TUC*_True.csv"))
mea_tuc_path = (
    mea_tuc_candidates[0]
    if len(mea_tuc_candidates) == 1
    else RAW_DATA_DIR / "MEA_TUC_FILE_NOT_FOUND.csv"
)
RAW_FILES = {
    "pea_bfkt": BACKEND_ROOT / "app" / "ml" / "data" / "PEA_2019-2026_BFKT_True.csv",
    "pea_tuc":  BACKEND_ROOT / "app" / "ml" / "data" / "PEA_2019-2026_TUC_True.csv",
    "mea_bfkt": BACKEND_ROOT / "app" / "ml" / "data" / "MEA_2019-2026_BFKT(TUC)_True.csv",
    "mea_tuc":  mea_tuc_path,
    "mea_tmv":  BACKEND_ROOT / "app" / "ml" / "data" / "MEA_2019-2026_TMV(TUC)_True.csv",
}

# Set either range to a (start_YYYYMM, end_YYYYMM) tuple to override.
# Raw-file mode otherwise reserves the latest four complete months for
# duration follow-up, uses the preceding 12 months for testing, and up to
# 36 earlier months for training.
REQUESTED_TRAIN_RANGE = None
REQUESTED_TEST_RANGE = None

QUANTILES = QuantileConfig(q_low=0.05, q_mid=0.50, q_high=0.95)
THRESHOLDS = ClassifyThresholds(up=1.5, down=1 / 1.5, sustain=1.3)
EVENT_CONFIG = SeverityDurationConfig(
    severity_medium_min=1.0,
    severity_high_min=3.0,
    baseline_months=4,
    min_baseline_months=4,
    up_ratio=THRESHOLDS.up,
    elevated_ratio=THRESHOLDS.sustain,
    long_duration_months=4,
)
EXPORT_RESULTS = False
OUTPUT_DIR = BACKEND_ROOT / "notebooks" / "outputs"

print("Quantiles:", asdict(QUANTILES))
print("Classification thresholds:", asdict(THRESHOLDS))
print("Event thresholds:", asdict(EVENT_CONFIG))

In [ ]:
def _yyyymm(period: pd.Period) -> int:
    return period.year * 100 + period.month


def _automatic_ranges(master: pd.DataFrame) -> tuple[tuple[int, int], tuple[int, int]]:
    months = pd.PeriodIndex(
        pd.to_datetime(master["month"].astype(str), format="%Y%m"), freq="M"
    ).sort_values().unique()
    if len(months) < 24:
        raise ValueError("At least 24 complete calendar months are needed for an automatic split.")
    test_end = months[-1] - 4
    test_start = test_end - 11
    train_end = test_start - 1
    train_start = max(months[0], train_end - 35)
    return (
        (_yyyymm(train_start), _yyyymm(train_end)),
        (_yyyymm(test_start), _yyyymm(test_end)),
    )


def make_synthetic_container(seed: int = 42) -> tuple[DataBillContainer, pd.DataFrame]:
    """Deterministic site-month demo with planted 1/2/3/5-month rises."""
    rng = np.random.default_rng(seed)
    periods = pd.period_range("2019-01", "2025-12", freq="M")
    severity_design = [("smaller", 1.60), ("medium", 2.40), ("larger", 4.00)]
    duration_design = [1, 2, 3, 5]
    event_specs = []
    event_site_no = 0
    for design_severity, multiplier in severity_design:
        for duration in duration_design:
            for replicate in range(3):
                event_specs.append({
                    "site_id": f"SYN{event_site_no:04d}",
                    "start_month": pd.Period("2024-03", freq="M") + 2 * replicate,
                    "planted_duration": duration,
                    "planted_multiplier": multiplier,
                    "design_severity": design_severity,
                })
                event_site_no += 1

    specs_by_site = {spec["site_id"]: spec for spec in event_specs}
    site_ids = list(specs_by_site) + [f"SYN{i:04d}" for i in range(event_site_no, event_site_no + 90)]
    rows = []
    for site_no, site_id in enumerate(site_ids):
        base = rng.uniform(700, 1800)
        phase = rng.uniform(0, 2 * np.pi)
        seasonal = 0.08 * np.sin(2 * np.pi * np.arange(len(periods)) / 12 + phase)
        # Volatile training history gives the quantile band realistic width;
        # quieter holdout months keep the planted event onsets unambiguous.
        noise = rng.normal(0, 0.18, len(periods))
        holdout_mask = periods >= pd.Period("2024-01", freq="M")
        noise[holdout_mask] = rng.normal(0, 0.04, int(holdout_mask.sum()))
        values = np.clip(base * (1 + seasonal + noise), 0.35 * base, None)

        if site_id in specs_by_site:
            spec = specs_by_site[site_id]
            start = periods.get_loc(spec["start_month"])
            baseline = float(np.median(values[start - 4:start]))
            stop = start + spec["planted_duration"]
            values[start:stop] = baseline * spec["planted_multiplier"]
            values[stop] = baseline * 0.95  # observed return closes the event

        for period, kwh in zip(periods, values):
            rows.append({
                "provider": "PEA" if site_no % 2 == 0 else "MEA",
                "company": "TUC",
                "Meter_No": str(100000 + site_no),
                "Site_ID": site_id,
                "site_type": "NORMAL",
                "Rate_CAT": "DEMO",
                "TOU_TOD": "TOU",
                "Province": "BKK",
                "month": _yyyymm(period),
                "date": period.to_timestamp(),
                "bill_amount": float(kwh * 4.2),
                "kwh": float(kwh),
                "bill_class": "active",
                "is_shutdown": False,
            })

    container = DataBillContainer()
    container.master_df = pd.DataFrame(rows)
    return container, pd.DataFrame(event_specs)


## 2. Load and preprocess

Raw mode performs the complete five-file PEA/MEA ingestion. The production master builder intentionally drops the globally latest month because that billing cycle is usually incomplete; this also means end-of-history duration must be treated carefully.

In [ ]:
raw_files_available = all(Path(path).is_file() for path in RAW_FILES.values())
if DATA_MODE == "auto":
    RUN_MODE = "raw_files" if raw_files_available else "synthetic"
elif DATA_MODE in {"raw_files", "synthetic"}:
    RUN_MODE = DATA_MODE
else:
    raise ValueError("DATA_MODE must be 'auto', 'synthetic', or 'raw_files'.")

if RUN_MODE == "raw_files":
    missing_paths = {key: path for key, path in RAW_FILES.items() if not Path(path).is_file()}
    if missing_paths:
        formatted = "\n".join(f"  {key}: {path}" for key, path in missing_paths.items())
        raise FileNotFoundError(f"Edit RAW_FILES; these inputs do not exist:\n{formatted}")
    container = DataBillContainer()
    container.load_files({key: str(path) for key, path in RAW_FILES.items()})
    master = container.ensure_master()
    planted_events = pd.DataFrame()
    drop_options = DropOptions(
        duplicate_site=True, common_site=True, shutdown_site=True, maintenance_site=True
    )
elif RUN_MODE == "synthetic":
    container, planted_events = make_synthetic_container()
    master = container.ensure_master()
    drop_options = DropOptions()
auto_train, auto_test = _automatic_ranges(master)
if RUN_MODE == "synthetic":
    auto_train, auto_test = (201901, 202312), (202401, 202412)
train_months = REQUESTED_TRAIN_RANGE or auto_train
test_months = REQUESTED_TEST_RANGE or auto_test
train_range = DateRange(*train_months)
test_range = DateRange(*test_months)
if train_range.end_month >= test_range.start_month:
    raise ValueError("Use chronological non-overlapping windows: train_end < test_start.")

print(f"Mode: {RUN_MODE} (requested: {DATA_MODE})")
print(f"Master rows/sites/months: {len(master):,} / {master['Site_ID'].nunique():,} / {master['month'].nunique():,}")
print(f"Available: {int(master['month'].min())} to {int(master['month'].max())}")
print(f"Train: {train_months} | Test: {test_months}")
if RUN_MODE == "raw_files":
    print(f"Latest incomplete month dropped by preprocessing: {container.dropped_latest_month}")

In [ ]:
duplicate_site_months = int(master.duplicated(["Site_ID", "month"]).sum())
zero_or_missing_kwh = int((master["kwh"].isna() | master["kwh"].fillna(0).eq(0)).sum())
print(f"Duplicate site-month rows before selected drop options: {duplicate_site_months:,}")
print(f"Zero/missing kWh rows before selected drop options: {zero_or_missing_kwh:,}")

preview = preview_missing_rate(
    container, drop_options, train_range.start_month, test_range.end_month
)
print("Drop report:")
print(json.dumps(preview["drop_report"], indent=2))
print("Window missing-rate summary:")
print(json.dumps({k: v for k, v in preview["missing"].items() if k != "per_month"}, indent=2))

## 3. Run the production quantile ML and spike/step classifier

The model predicts next-month kWh. `bill_month` is the predictor row and `anom_m = bill_month + 1` is the actual unusual month used for event duration. We explicitly use `.05/.50/.95`; the Python dataclass's bare defaults differ from the API defaults.

In [ ]:
state = MLRunState()  # fresh notebook-local state; never reuse the API global
build_summary = build_pipeline(
    container, drop_options, train_range, test_range, QUANTILES, state
)
classified = classify_pipeline(state, THRESHOLDS)
abnormal = abnormal_dataframe(state)

print("Build summary:")
print(json.dumps(build_summary, indent=2, default=str))
print(f"Flagged site-months: {len(state.test_flagged):,}")
print("Classifier counts:")
print(classified["anom_type"].value_counts(dropna=False).to_string())
if not classified.empty:
    preview_cols = ["site_id", "bill_month", "anom_m", "anom_val", "anom_type", "quantile_severity"]
    print("\nMost severe classified rows:")
    print(classified.sort_values("quantile_severity", ascending=False)[preview_cols].head(10).to_string(index=False))

## 4. Verify and use severity

The score is not an arbitrary 1-10 label. A score of `0.5` is half a learned prediction-band width above q95; `2.0` is two widths above it. We recompute the formula below to prove the notebook and backend use the same quantity.

In [ ]:
if state.test_flagged.empty:
    print("The selected run produced no upper-tail flags; choose a wider test range or inspect model coverage.")
else:
    recomputed = recompute_quantile_severity(state.test_flagged)
    np.testing.assert_allclose(
        recomputed.to_numpy(), state.test_flagged["quantile_severity"].to_numpy(), atol=1e-12
    )
    print("Severity formula check: PASS (exactly matches backend values)")
    print(state.test_flagged["quantile_severity"].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).to_string())

print("Severity bins: Low < 1; Medium 1 to < 3; High >= 3 band widths above q95.")

## 5. Measure consecutive duration and collapse flags into events

For each surfaced upward anomaly, `anom_m` is the detection month. The duration clock walks backward to an earlier qualifying jump when the series stayed continuously elevated through detection; otherwise it starts at `anom_m`. The baseline is the median of exactly four complete prior calendar months. The run stays open while each consecutive month remains at least `sustain x baseline`. Later flagged months inside that run are merged into the same event.

In [ ]:
events = build_severity_duration_events(classified, state.full_history, EVENT_CONFIG)
print(f"De-duplicated upward events: {len(events):,}")
if events.empty:
    print("No spike_up/step_up events cleared the selected thresholds.")
else:
    print("Duration audit status:")
    print(events["duration_status"].value_counts(dropna=False).to_string())
    print(f"Confirmed for matrix: {int(events['duration_confirmed'].sum()):,}")
    event_cols = [
        "event_id", "detection_month", "start_month", "months_before_detection",
        "end_month", "baseline_kwh",
        "onset_ratio", "duration_months", "duration_band",
        "detection_quantile_severity", "severity_band",
        "detection_anom_type", "right_censored", "n_flagged_months",
    ]
    print("\nEvent sample:")
    print(events[event_cols].head(15).to_string(index=False))

## 6. The 9-cell matrix

Each cell is an event count. The percentage in parentheses is within its duration row, which makes severity mix comparable even when duration groups have different sizes. Empty categories remain visible, so the result is always exactly 3 x 3.

In [ ]:
matrix_counts = severity_duration_matrix(events)
matrix_row_pct = severity_duration_matrix(events, row_percent=True)
print("Event counts:")
print(matrix_counts.to_string())
print("\nWithin-duration row percentages:")
print(matrix_row_pct.to_string())

fig, ax = plt.subplots(figsize=(8.5, 4.8))
image = ax.imshow(matrix_counts.to_numpy(), cmap="YlOrRd", aspect="auto")
for row in range(len(DURATION_BANDS)):
    for col in range(len(SEVERITY_BANDS)):
        count = int(matrix_counts.iloc[row, col])
        pct = float(matrix_row_pct.iloc[row, col])
        ax.text(col, row, f"{count:,}\n({pct:.1f}%)", ha="center", va="center", color="black", fontweight="bold")
ax.set_xticks(range(len(SEVERITY_BANDS)), SEVERITY_BANDS)
ax.set_yticks(range(len(DURATION_BANDS)), DURATION_BANDS)
ax.set_xlabel("Quantile severity")
ax.set_ylabel("Observed elevated duration")
ax.set_title("Anomaly events: severity x duration")
fig.colorbar(image, ax=ax, label="Event count")
fig.tight_layout()
if "ipykernel" in sys.modules:
    plt.show()
else:
    fig.canvas.draw()  # headless validation path
plt.close(fig)

## 7. Does observed duration agree with spike/step intuition?

The check below asks whether a confirmed single-month event was labelled `spike_up`, and whether confirmed 2-3 or `>=4` month events were labelled `step_up` by the existing four-month post-median classifier. Rates include 95% Wilson intervals so a tiny sample is not mistaken for strong evidence. Disagreements are expected where the median heuristic and exact consecutive duration encode different ideas.

In [ ]:
intuition = duration_intuition_report(events)
type_table = duration_type_crosstab(events)
print("Duration versus current classifier:")
print(type_table.to_string())
print("\nConsistency rates (rate and 95% Wilson interval):")
print(json.dumps(intuition, indent=2))

eligible = events[events["duration_confirmed"].fillna(False) & events["duration_band"].notna()].copy()
single = eligible[eligible["duration_band"] == DURATION_BANDS[0]]
multi = eligible[eligible["duration_band"].isin(DURATION_BANDS[1:])]
single_rate = (single["detection_anom_type"] == "spike_up").mean() if len(single) else np.nan
multi_rate = (multi["detection_anom_type"] == "step_up").mean() if len(multi) else np.nan
scope = "SYNTHETIC DEMO ONLY" if RUN_MODE == "synthetic" else "LOADED BILLING DATA"
print(f"\nEvidence scope: {scope}")
print(f"P(current label = spike_up | observed single month) = {single_rate:.1%} (n={len(single)})" if len(single) else "No confirmed single-month events.")
print(f"P(current label = step_up | observed >=2 months) = {multi_rate:.1%} (n={len(multi)})" if len(multi) else "No confirmed multi-month events.")
if len(single) < 5 or len(multi) < 5:
    print("Conclusion: sample is too small for a stable intuition check; collect more confirmed events.")
elif single_rate >= 0.8 and multi_rate >= 0.8:
    print("Conclusion: the loaded events are strongly consistent with single=spike and multi=step.")
else:
    print("Conclusion: evidence is mixed; exact duration and the current four-month-median labels should not be treated as interchangeable.")

In [ ]:
if RUN_MODE == "synthetic":
    planted_check = planted_events.merge(
        events[["site_id", "start_month", "duration_months", "duration_band", "severity_band"]],
        on=["site_id", "start_month"],
        how="left",
        indicator=True,
    )
    detected = planted_check["_merge"].eq("both")
    duration_matches = planted_check.loc[detected, "duration_months"].eq(
        planted_check.loc[detected, "planted_duration"]
    )
    print(f"Planted onsets surfaced as events: {int(detected.sum())}/{len(planted_check)}")
    print(f"Exact duration recovery among surfaced planted events: {duration_matches.mean():.1%}" if detected.any() else "No planted events surfaced.")
    print(planted_check.head(12).to_string(index=False))

## 8. Optional reproducible exports

Set `EXPORT_RESULTS = True` in the configuration cell to write the event audit, count matrix, row percentages, and the thresholds/date metadata used for the run.

In [ ]:
if EXPORT_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    export_events = events.copy()
    for column in ("detection_month", "start_month", "end_month", "first_return_month"):
        export_events[column] = export_events[column].astype("string")
    export_events.to_csv(OUTPUT_DIR / "severity_duration_events.csv", index=False)
    matrix_counts.to_csv(OUTPUT_DIR / "severity_duration_matrix_counts.csv")
    matrix_row_pct.to_csv(OUTPUT_DIR / "severity_duration_matrix_row_percent.csv")
    metadata = {
        "data_mode": RUN_MODE,
        "train_range": list(train_months),
        "test_range": list(test_months),
        "quantiles": asdict(QUANTILES),
        "classification_thresholds": asdict(THRESHOLDS),
        "event_config": asdict(EVENT_CONFIG),
        "intuition_report": intuition,
    }
    (OUTPUT_DIR / "severity_duration_run_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    print(f"Wrote analysis outputs to {OUTPUT_DIR}")
else:
    print("Exports are off. Set EXPORT_RESULTS = True to save the audit artifacts.")

## Reading the result

Use the 3 x 3 table for prioritization, not as a causal label:

- Move **right** when the observed kWh is farther beyond what the quantile model expected.
- Move **down** when the elevated level persists for more consecutive months.
- Inspect the event audit before acting on sparse cells, censoring, or disagreements. A high-severity single month is a strong transient spike candidate; a long-duration event is a sustained level-shift candidate, but billing corrections, site consolidation, and missing reads still require domain review.
- For production conclusions, switch to `raw_files`, provide all five current exports, and retain the exported metadata with the matrix so thresholds and date windows remain auditable.